In [ ]:
from google.colab import drive
drive.mount("/content/drive")



Mounted at /content/drive


In [ ]:
import math
from pathlib import Path

import gradio as gr
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageFont
from torch import nn
from torchvision import models, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DRIVE_CANDIDATES = [
    Path('/content/drive/MyDrive/NatDisaster'),
    Path('/drive/NatDisaster'),
]
DRIVE_DIR = next((path for path in DRIVE_CANDIDATES if path.exists()), DRIVE_CANDIDATES[0])

MODELS_DIR = DRIVE_DIR / 'models_main'
ROAD_MODELS_DIR = DRIVE_DIR / 'models_roads'
LARGE_MODELS_DIR = DRIVE_DIR / 'models_largeareas'

MODEL_CANDIDATES = [
    MODELS_DIR / 'cnn_convnext_tiny_natdisaster.pth',
    MODELS_DIR / 'cnn_convnext_tiny_best.pt',
    DRIVE_DIR / 'cnn_convnext_tiny_best.pt',
    Path('/content/cnn_convnext_tiny_best.pt'),
]

ROAD_MODEL_CANDIDATES = [
    ROAD_MODELS_DIR / 'resnet50_roads_best.pth',
    Path('/drive/NatDisaster/models_roads/resnet50_roads_best.pth'),
    Path('/content/drive/MyDrive/NatDisaster/models_roads/resnet50_roads_best.pth'),
]

LARGE_AREA_MODEL_CANDIDATES = [
    LARGE_MODELS_DIR / 'resnet50_largeareas_best.pth',
    Path('/drive/NatDisaster/models_largeareas/resnet50_largeareas_best.pth'),
    Path('/content/drive/MyDrive/NatDisaster/models_largeareas/resnet50_largeareas_best.pth'),
]

MODEL_PATH = next((path for path in MODEL_CANDIDATES if path.exists()), None)
ROAD_MODEL_PATH = next((path for path in ROAD_MODEL_CANDIDATES if path.exists()), None)
LARGE_AREA_MODEL_PATH = next((path for path in LARGE_AREA_MODEL_CANDIDATES if path.exists()), None)

In [ ]:
def build_cnn(num_classes=2):
    model = models.convnext_tiny(weights=None)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model

def build_classifier_model(model_name, num_classes=2):
    if model_name == 'resnet50':
        model = models.resnet50(weights=None)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
    elif model_name == 'convnext_tiny':
        model = models.convnext_tiny(weights=None)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model

def get_positive_index(checkpoint, positive_class_name, fallback=1):
    class_to_idx = checkpoint.get('class_to_idx') or checkpoint.get('class_to_label') or {}
    if positive_class_name in class_to_idx:
        return int(class_to_idx[positive_class_name])

    class_names = checkpoint.get('class_names') or list(class_to_idx.keys())
    if positive_class_name in class_names:
        return int(class_names.index(positive_class_name))

    return fallback

def make_eval_transform_for_checkpoint(checkpoint):
    size = int(checkpoint.get('image_size', 224))
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])




In [ ]:


checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
image_size = int(checkpoint.get('image_size', 224))
class_to_label = checkpoint.get('class_to_label') or checkpoint.get('class_to_idx') or {'Normal': 0, 'Disaster': 1}
disaster_index = int(class_to_label.get('Disaster', 1))

model = build_cnn(num_classes=2)
model.load_state_dict(checkpoint['state_dict'])
model.to(device)
model.eval()

eval_transform = transforms.Compose([
    transforms.Resize(int(image_size * 1.14)),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

road_checkpoint = torch.load(ROAD_MODEL_PATH, map_location='cpu', weights_only=False)
road_class_names = road_checkpoint.get('class_names', ['RoadClear', 'RoadBlocked'])
road_model = build_classifier_model(road_checkpoint.get('model_name', 'resnet50'), len(road_class_names))
road_model.load_state_dict(road_checkpoint['state_dict'])
road_model.to(device)
road_model.eval()
road_transform = make_eval_transform_for_checkpoint(road_checkpoint)
road_blocked_index = get_positive_index(road_checkpoint, 'RoadBlocked', fallback=1)

large_area_checkpoint = torch.load(LARGE_AREA_MODEL_PATH, map_location='cpu', weights_only=False)
large_area_class_names = large_area_checkpoint.get('class_names', ['LargeAreas', 'NotLargeAreas'])
large_area_model = build_classifier_model(large_area_checkpoint.get('model_name', 'resnet50'), len(large_area_class_names))
large_area_model.load_state_dict(large_area_checkpoint['state_dict'])
large_area_model.to(device)
large_area_model.eval()
large_area_transform = make_eval_transform_for_checkpoint(large_area_checkpoint)
large_area_index = get_positive_index(large_area_checkpoint, 'LargeAreas', fallback=0)

In [ ]:
rows =3
cols = 4
num_cells = 12

def split_into_grid(image, rows=rows, cols=cols):
    image = image.convert('RGB')
    width, height = image.size
    cells = []

    for r in range(rows):
        for c in range(cols):
            left = round(c * width / cols)
            right = round((c + 1) * width / cols)
            top = round(r * height / rows)
            bottom = round((r + 1) * height / rows)
            crop = image.crop((left, top,right, bottom))
            cell_id = r * cols + c + 1
            cells.append({
                'grid': cell_id,
                'row': r+1,
                'col': c+1,
                'box': (left, top, right, bottom),
                'crop': crop
            })
    return cells


In [ ]:
@torch.no_grad()
def predict_disaster_scores(crops):
    tensors = [eval_transform(crop.convert('RGB')) for crop in crops]
    batch = torch.stack(tensors).to(device)
    logits = model(batch)
    probs = torch.softmax(logits, dim=1)[:, disaster_index]
    return probs.detach().cpu().numpy()



In [ ]:
@torch.no_grad()
def predict_positive_scores(crops, model_obj, transform_obj, positive_index, batch_size=32):
    if not crops:
        return np.array([], dtype=np.float32)

    all_scores = []
    for start in range(0, len(crops), batch_size):
        batch_crops = crops[start:start + batch_size]
        tensors = [transform_obj(crop.convert('RGB')) for crop in batch_crops]
        batch = torch.stack(tensors).to(device)
        logits = model_obj(batch)
        probs = torch.softmax(logits, dim=1)[:, positive_index]
        all_scores.extend(probs.detach().cpu().numpy().tolist())

    return np.array(all_scores, dtype=np.float32)

def predict_road_blocked_scores(crops):
    return predict_positive_scores(crops, road_model, road_transform, road_blocked_index)

def predict_large_area_scores(crops):
    return predict_positive_scores(crops, large_area_model, large_area_transform, large_area_index)

def normalize_population(populations):
    populations = np.array(populations, dtype=np.float32)
    max_pop = populations.max() if len(populations) else 0
    if max_pop <= 0:
        return np.zeros_like(populations)
    return populations / max_pop


In [ ]:
ROAD_WARNING_THRESHOLD = 0.5
LANDING_AREA_THRESHOLD = 0.70


def sliding_windows(image, window_fracs=(0.35, 0.50), overlap=0.50):
    image = image.convert('RGB')
    width, height = image.size
    min_side = min(width, height)
    boxes = []

    for frac in window_fracs:
        side = max(96, int(min_side * frac))
        side = min(side, width, height)
        step = max(48, int(side * (1 - overlap)))

        xs = list(range(0, max(1, width - side + 1), step))
        ys = list(range(0, max(1, height - side + 1), step))

        if xs[-1] != width - side:
            xs.append(width - side)
        if ys[-1] != height - side:
            ys.append(height - side)

        for y in ys:
            for x in xs:
                box = (int(x), int(y), int(x + side), int(y + side))
                if box not in boxes:
                    boxes.append(box)

    return boxes


def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    intersection = iw * ih

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - intersection

    if union <= 0:
        return 0.0
    return intersection / union



In [ ]:
def non_max_suppression(detections, iou_threshold=0.25, max_boxes=6):
    detections = sorted(detections, key=lambda item: item['score'], reverse=True)
    kept = []

    for det in detections:
        if all(box_iou(det['box'], kept_det['box']) < iou_threshold for kept_det in kept):
            kept.append(det)
        if len(kept) >= max_boxes:
            break

    return kept


def detect_blocked_roads(image):
    boxes = sliding_windows(image, window_fracs=(0.35, 0.50), overlap=0.50)
    crops = [image.crop(box) for box in boxes]
    scores = predict_road_blocked_scores(crops)

    if len(scores) == 0:
        return 0.0, []

    detections = [
        {'box': box, 'score': float(score)}
        for box, score in zip(boxes, scores)
    ]

    top_scores = sorted(scores, reverse=True)[:3]
    overall_probability = 0.70 * float(np.max(scores)) + 0.30 * float(np.mean(top_scores))
    road_detections = non_max_suppression(
        [det for det in detections if det['score'] >= ROAD_WARNING_THRESHOLD],
        iou_threshold=0.30,
        max_boxes=5,
    )

    return min(1.0, overall_probability), road_detections

In [ ]:
def detect_large_areas(image):
    boxes = sliding_windows(image, window_fracs=(0.24, 0.34, 0.44), overlap=0.55)
    crops = [image.crop(box) for box in boxes]
    scores = predict_large_area_scores(crops)

    detections = [
        {'box': box, 'score': float(score)}
        for box, score in zip(boxes, scores)
    ]

    strong = [det for det in detections if det['score'] >= LANDING_AREA_THRESHOLD]
    if not strong:
        strong = [det for det in sorted(detections, key=lambda item: item['score'], reverse=True)[:3] if det['score'] >= 0.50]

    return non_max_suppression(strong, iou_threshold=0.25, max_boxes=6)


In [ ]:
def draw_landing_area_boxes(image, detections):
    image = image.convert('RGB')
    overlay = image.copy()
    draw = ImageDraw.Draw(overlay, 'RGBA')

    try:
        box_font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 24)
    except:
        box_font = ImageFont.load_default()

    for i, det in enumerate(detections, start=1):
        left, top, right, bottom = det['box']
        score = det['score']

        draw.rectangle((left, top, right, bottom), outline=(0, 255, 110, 255), width=5)
        label = f'Landing {i}: {score * 100:.0f}%'
        bbox = draw.textbbox((0, 0), label, font=box_font)
        label_w = bbox[2] - bbox[0]
        label_h = bbox[3] - bbox[1]
        label_y = max(0, top - label_h - 12)

        draw.rectangle((left, label_y, left + label_w + 16, label_y + label_h + 10), fill=(0, 80, 35, 210))
        draw.text((left + 8, label_y + 5), label, fill=(255, 255, 255, 255), font=box_font)

    return overlay

In [ ]:
def draw_grid_preview(image):
    if image is None:
        raise gr.Error("Please upload an image first.")

    image = image.convert("RGB")
    overlay = image.copy()
    draw = ImageDraw.Draw(overlay, "RGBA")
    width, height = image.size

    try:
        grid_font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
            32
        )
    except:
        grid_font = ImageFont.load_default()

    for r in range(rows):
        for c in range(cols):
            grid_id = r * cols + c + 1

            left = round(c * width / cols)
            right = round((c + 1) * width / cols)
            top = round(r * height / rows)
            bottom = round((r + 1) * height / rows)

            draw.rectangle(
                (left, top, right, bottom),
                outline=(255, 255, 255, 240),
                width=5
            )
            label = str(grid_id)
            bbox = draw.textbbox((0, 0), label, font=grid_font)
            text_w = bbox[2] - bbox[0]
            text_h = bbox[3] - bbox[1]

            pad_x = 12
            pad_y = 8
            x = left + 12
            y = top + 12

            draw.rectangle(
                (
                    x - pad_x,
                    y - pad_y,
                    x + text_w + pad_x,
                    y + text_h + pad_y,
                ),
                fill=(0, 0, 0, 170)
            )

            draw.text(
                (x, y),
                label,
                fill=(255, 255, 255, 255),
                font=grid_font
            )

    return overlay

In [ ]:
def draw_grid_overlay(image, results_df=None):
    image = image.convert("RGB")
    overlay = image.copy()
    draw = ImageDraw.Draw(overlay, "RGBA")
    width, height = image.size

    try:
        grid_font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
            28
        )
        rank_font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
            44
        )
    except:
        grid_font = ImageFont.load_default()
        rank_font = ImageFont.load_default()

    rank_by_grid = {}
    urgency_by_grid = {}

    if results_df is not None and len(results_df):
        rank_by_grid = dict(zip(results_df["grid"], results_df["rank"]))
        urgency_by_grid = dict(zip(results_df["grid"], results_df["urgency_score"]))

    max_urgency = max(urgency_by_grid.values()) if urgency_by_grid else 1.0

    for r in range(rows):
        for c in range(cols):
            grid_id = r * cols + c + 1

            left = round(c * width / cols)
            right = round((c + 1) * width / cols)
            top = round(r * height / rows)
            bottom = round((r + 1) * height / rows)

            urgency = urgency_by_grid.get(grid_id, 0.0)
            alpha = int(40 + 135 * (urgency / max_urgency)) if urgency_by_grid else 0

            if urgency_by_grid:
                draw.rectangle(
                    (left, top, right, bottom),
                    fill=(255, 30, 30, alpha)
                )

            draw.rectangle(
                (left, top, right, bottom),
                outline=(255, 255, 255, 230),
                width=4
            )

            # Top-left grid number
            grid_label = str(grid_id)
            grid_bbox = draw.textbbox((0, 0), grid_label, font=grid_font)
            grid_text_w = grid_bbox[2] - grid_bbox[0]
            grid_text_h = grid_bbox[3] - grid_bbox[1]

            grid_pad_x = 10
            grid_pad_y = 6
            grid_x = left + 10
            grid_y = top + 10

            draw.rectangle(
                (
                    grid_x - grid_pad_x,
                    grid_y - grid_pad_y,
                    grid_x + grid_text_w + grid_pad_x,
                    grid_y + grid_text_h + grid_pad_y,
                ),
                fill=(0, 0, 0, 160)
            )

            draw.text(
                (grid_x, grid_y),
                grid_label,
                fill=(255, 255, 255, 255),
                font=grid_font
            )

            if grid_id in rank_by_grid:
                rank_label = f"Rank{rank_by_grid[grid_id]}"

                rank_bbox = draw.textbbox((0, 0), rank_label, font=rank_font)
                rank_text_w = rank_bbox[2] - rank_bbox[0]
                rank_text_h = rank_bbox[3] - rank_bbox[1]

                center_x = (left + right) / 2
                center_y = (top + bottom) / 2

                rank_x = center_x - rank_text_w / 2
                rank_y = center_y - rank_text_h / 2

                rank_pad_x = 14
                rank_pad_y = 8

                draw.rectangle(
                    (
                        rank_x - rank_pad_x,
                        rank_y - rank_pad_y,
                        rank_x + rank_text_w + rank_pad_x,
                        rank_y + rank_text_h + rank_pad_y,
                    ),
                    fill=(0, 0, 0, 150)
                )

                draw.text(
                    (rank_x, rank_y),
                    rank_label,
                    fill=(255, 255, 255, 255),
                    font=rank_font
                )

    return overlay

In [ ]:
def calculate_grid_rankings(
    image,
    pop_1, pop_2, pop_3, pop_4,
    pop_5, pop_6, pop_7, pop_8,
    pop_9, pop_10, pop_11, pop_12,
    image_weight,
    population_weight,
):

    if image is None:
        raise gr.Error('please upload an image first')

    populations = [
        pop_1, pop_2, pop_3, pop_4,
        pop_5, pop_6, pop_7, pop_8,
        pop_9, pop_10, pop_11, pop_12,
    ]
    populations = [0 if value is None else max(0, float(value)) for value in populations]

    total_weight = image_weight + population_weight
    if total_weight <= 0:
        raise gr.Error('Cannot be negative')

    cells = split_into_grid(image)
    crops = [cell['crop'] for cell in cells]
    disaster_scores = predict_disaster_scores(crops)
    population_scores = normalize_population(populations)
    urgency_scores = 0.65 * disaster_scores + 0.35 * population_scores

    rows = []
    for cell, population, pop_score, disaster_score, urgency_score in zip(
        cells, populations, population_scores, disaster_scores, urgency_scores
    ):
        rows.append({
            'grid': cell['grid'],
            'row': cell['row'],
            'col': cell['col'],
            'population': int(population),
            'population_score': round(float(pop_score), 4),
            'disaster_score': round(float(disaster_score), 4),
            'urgency_score': round(float(urgency_score), 4),
        })

    results_df = pd.DataFrame(rows)
    results_df = results_df.sort_values('urgency_score', ascending=False).reset_index(drop=True)
    results_df.insert(0, 'rank', np.arange(1, len(results_df) + 1))

    overlay = draw_grid_overlay(image, results_df)

    top_grid = int(results_df.iloc[0]['grid'])
    top_score = float(results_df.iloc[0]['urgency_score'])
    summary = f'Highest priority: Grid {top_grid} with urgency score {top_score:.4f}'

    road_probability, road_detections = detect_blocked_roads(image)
    road_percent = road_probability * 100

    if road_probability >= ROAD_WARNING_THRESHOLD:
        landing_detections = detect_large_areas(image)
        landing_overlay = draw_landing_area_boxes(image, landing_detections)

        if landing_detections:
            road_message = (
                f'<div style="font-size: 20px; line-height: 1.45;">'
                f'Watch out! There is a <strong style="color: #dc2626;">{road_percent:.1f}%</strong> chance '
                f'the area is inaccessible due to blocked roads. '
                f'Here are {len(landing_detections)} possible helicopter landing locations or staging areas.'
                f'</div>'
            )
        else:
            road_message = (
                f'<div style="font-size: 20px; line-height: 1.45;">'
                f'Watch out! There is a <strong style="color: #dc2626;">{road_percent:.1f}%</strong> chance '
                f'the area is inaccessible due to blocked roads. '
                f'No strong helicopter landing areas were found in this image.'
                f'</div>'
            )
    else:
        landing_detections = []
        landing_overlay = image.convert('RGB')
        road_message = (
            f'<div style="font-size: 18px; line-height: 1.45;">'
            f'No major blocked-road warning detected. Highest blocked-road score: '
            f'<strong>{road_percent:.1f}%</strong>.'
            f'</div>'
        )

    return overlay, results_df, summary, road_message, landing_overlay

In [ ]:
with gr.Blocks(title='Natural Disaster First Responder Management') as demo:
    gr.Markdown('# Natural Disaster 12-Grid First Responder Management')
    gr.Markdown('Upload one aerial image, preview the 12-cell grid, enter population for each grid cell, then calculate priority rankings.')

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type='pil', label='Aerial image')

            matrix_button = gr.Button('Preview Sections')

            gr.Markdown('### Population Inputs')
            gr.Markdown('Grid numbering: 1-4 top row, 5-8 middle row, 9-12 bottom row.')

            pop_inputs = []
            for row_start in [1, 5, 9]:
                with gr.Row():
                    for grid_id in range(row_start, row_start + 4):
                        pop_inputs.append(
                            gr.Number(
                                value=0,
                                minimum=0,
                                label=f'Grid {grid_id} pop',
                            )
                        )

            calculate_button = gr.Button('Calculate Rankings', variant='primary')

            image_weight = gr.Number(value=0.65, visible=False)
            population_weight = gr.Number(value=0.35, visible=False)

        with gr.Column(scale=1):
            matrix_preview_output = gr.Image(type='pil', label='Grid preview')
            overlay_output = gr.Image(type='pil', label='Ranked grid visualization')
            summary_output = gr.Markdown(label='Summary')
            table_output = gr.Dataframe(label='Ranked grid results', wrap=True)
            road_warning_output = gr.Markdown(label='Road access warning')
            landing_output = gr.Image(type='pil', label='Possible helicopter landing areas')

    matrix_button.click(
        fn=draw_grid_preview,
        inputs=[image_input],
        outputs=[matrix_preview_output],
    )

    calculate_button.click(
        fn=calculate_grid_rankings,
        inputs=[image_input] + pop_inputs + [image_weight, population_weight],
        outputs=[overlay_output, table_output, summary_output, road_warning_output, landing_output],
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://35a46a82fadca22e83.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
